In [32]:
import pandas as pd
import re
import nltk
import numpy as np
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.utils import resample
from gensim.models import Word2Vec


In [33]:
print("Tahap 1: Preprocessing & Cleaning")
nltk.download('stopwords')
stop_words = set(stopwords.words('indonesian'))

Tahap 1: Preprocessing & Cleaning


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Vincent\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [34]:
slang_dict = {
    "cit": "curang", "cheat": "curang", "bngt": "banget", "bgtt": "banget",
    "mantap": "bagus", "mantapp": "bagus", "oke": "bagus", "ok": "bagus",
    "jelek": "buruk", "gblk": "bodoh", "tolol": "bodoh", "lola": "lambat",
    "u": "kamu", "lu": "kamu", "gw": "saya", "gua": "saya", "sy": "saya",
    "yg": "yang", "ga": "tidak", "gak": "tidak", "gk": "tidak", "udh": "sudah",
    "tp": "tapi", "klo": "kalau", "kalo": "kalau", "pas": "saat"
}

In [35]:
positive = set([
    'bagus', 'mantap', 'seru', 'oke', 'ok', 'keren', 'suka', 'puas', 
    'menyenangkan', 'hebat', 'asik', 'asah', 'cerdas', 'pintar', 'lancar', 
    'bermanfaat', 'membantu', 'top', 'sip', 'jos', 'enak', 'nyaman', 'mudah'
])

negative = set([
    'buruk', 'kecewa', 'lambat', 'lola', 'error', 'bug', 'curang', 
    'bodoh', 'tolol', 'parah', 'rugi', 'kesal', 'benci', 'sulit', 'susah', 
    'bosan', 'lelet', 'sampah', 'payah', 'kurang', 'nyesal', 'kecewa', 'jelek'
])

In [36]:
df = pd.read_csv('chess_reviews.csv')

In [58]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    fixed_words = [slang_dict.get(w, w) for w in words]
    clean_words = [w for w in fixed_words if w not in stop_words and len(w) > 2]
    return " ".join(clean_words)

In [59]:
def label(text):
    score = 0
    words = str(text).split()
    
    for word in words:
        if word in positive:
            score += 1
        elif word in negative:
            score -= 1
    if score > 0:
        return 'Positif'
    elif score < 0:
        return 'Negatif'
    else:
        return 'Netral'

In [60]:
df['content_cleaned'] = df['content'].apply(clean)
df = df.dropna(subset=['content_cleaned'])
df = df[df['content_cleaned'] != '']
df['label'] = df['content_cleaned'].apply(label)
print("Total Data: ", len(df))

Total Data:  13931


In [61]:
print("Tahap 2: Menyeimbangkan dataset")
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    df['content_cleaned'], 
    df['label'], 
    test_size=0.2, 
    random_state=42,
    stratify=df['label'] 
)

df_train = pd.DataFrame({'text': X_train_raw, 'label': y_train_raw})

df_neg = df_train[df_train.label == 'Negatif']
df_net = df_train[df_train.label == 'Netral']
df_pos = df_train[df_train.label == 'Positif']

n_target = max(len(df_neg), len(df_net), len(df_pos))

Tahap 2: Menyeimbangkan dataset


In [62]:
df_balanced = pd.concat([
    resample(df_neg, replace=True, n_samples=n_target, random_state=42),
    resample(df_net, replace=True, n_samples=n_target, random_state=42),
    resample(df_pos, replace=True, n_samples=n_target, random_state=42)
])

X_train_bal = df_balanced['text']
y_train_bal = df_balanced['label']

In [63]:
print("Tahap 3: Modeling")
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,3), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train_bal)
X_test_tfidf = tfidf.transform(X_test_raw)

Tahap 3: Modeling


In [64]:
print("SVM + TFIDF + 80/20")
model_svm = LinearSVC(C=0.01, random_state=42, max_iter=3000)
model_svm.fit(X_train_tfidf, y_train_bal)
print("SVM Training  :", accuracy_score(y_train_bal, model_svm.predict(X_train_tfidf))*100)
print("SVM Testing   :", accuracy_score(y_test_raw, model_svm.predict(X_test_tfidf))*100)

SVM + TFIDF + 80/20
SVM Training  : 96.34044640667157
SVM Testing   : 93.36203803372803


In [65]:
print("RF + TFIDF + 80/20")
model_rf = RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=2, random_state=42)
model_rf.fit(X_train_tfidf, y_train_bal)
print("RF Training   :", accuracy_score(y_train_bal, model_rf.predict(X_train_tfidf))*100)
print("RF Testing    :", accuracy_score(y_test_raw, model_rf.predict(X_test_tfidf))*100)

RF + TFIDF + 80/20
RF Training   : 96.85553102771645
RF Testing    : 93.54144241119484


In [75]:
print("DL + TFIDF + 80/20")
model_dl = MLPClassifier(hidden_layer_sizes=(64,32), alpha=0.1, max_iter=500, random_state=42)
model_dl.fit(X_train_tfidf, y_train_bal)
print("MLP Training  :", accuracy_score(y_train_bal, model_dl.predict(X_train_tfidf))*100)
print("MLP Testing   :", accuracy_score(y_test_raw, model_dl.predict(X_test_tfidf))*100)

DL + TFIDF + 80/20
MLP Training  : 100.0
MLP Testing   : 98.74416935773233


In [66]:
tokenized = [text.split() for text in X_train_bal]

w2v_model = Word2Vec(
    sentences=tokenized,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

def get_avg_w2v(text):
    words = str(text).split()
    vectors = [w2v_model.wv[w] for w in words if w in w2v_model.wv]
    if len(vectors) == 0:
        return np.zeros(100)
    return np.mean(vectors, axis=0)

In [67]:
X_train_w2v = np.array([get_avg_w2v(text) for text in X_train_bal])
X_test_w2v = np.array([get_avg_w2v(text) for text in X_test_raw])

In [74]:
print("RF + Word2Vec + 80/20")
model_rf2 = RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=2, random_state=42)
model_rf2.fit(X_train_w2v, y_train_bal)
print("RF Training   :", accuracy_score(y_train_bal, model_rf2.predict(X_train_w2v))*100)
print("RF Testing    :", accuracy_score(y_test_raw, model_rf2.predict(X_test_w2v))*100)

RF + Word2Vec + 80/20
RF Training   : 99.64189354917832
RF Testing    : 83.35127377108002


In [69]:
df_train_70, df_test_30 = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df['label']
)
df_neg70 = df_train_70[df_train_70.label == 'Negatif']
df_net70 = df_train_70[df_train_70.label == 'Netral']
df_pos70 = df_train_70[df_train_70.label == 'Positif']
n_target70 = max(len(df_neg70), len(df_net70), len(df_pos70))

In [70]:
df_bal70 = pd.concat([
    resample(df_neg70, replace=True, n_samples=n_target70, random_state=42),
    resample(df_net70, replace=True, n_samples=n_target70, random_state=42),
    resample(df_pos70, replace=True, n_samples=n_target70, random_state=42)
])
tfidf70 = TfidfVectorizer(max_features=10000, ngram_range=(1,3))
X_train_70 = tfidf70.fit_transform(df_bal70['content_cleaned'])
X_test_30 = tfidf70.transform(df_test_30['content_cleaned'])
y_train_70 = df_bal70['label']
y_test_30 = df_test_30['label']

In [71]:
print("RF + TFIDF + 70/30")
model_rf3 = RandomForestClassifier(n_estimators=150, max_depth=20, min_samples_leaf=2, random_state=42)
model_rf3.fit(X_train_70, y_train_70)
print("RF Training   :", accuracy_score(y_train_70, model_rf3.predict(X_train_70))*100)
print("RF Testing    :", accuracy_score(y_test_30, model_rf3.predict(X_test_30))*100)

RF + TFIDF + 70/30
RF Training   : 96.80457450386814
RF Testing    : 92.22488038277513


In [72]:
print("SVM + TFIDF + 70/30")
model_svm2 = LinearSVC(C=0.01, random_state=42,max_iter=3000)
model_svm2.fit(X_train_70, y_train_70)
print("SVM Training  :", accuracy_score(y_train_70, model_svm2.predict(X_train_70))*100)
print("SVM Testing   :", accuracy_score(y_test_30, model_svm2.predict(X_test_30))*100)

SVM + TFIDF + 70/30
SVM Training  : 96.05336921179504
SVM Testing   : 93.58851674641149


In [73]:
print("Tahap 4: Uji Coba Inference")

def prediksi_final(kalimat):
    bersih = clean(kalimat)
    vektor = tfidf.transform([bersih])
    return model_dl.predict(vektor)[0]

sampel = [
    "gamenya seru banget",
    "banyak cit curang",
    "lumayan buat ngisi waktu"
]

for s in sampel:
    print(f"Input: {s} -> Prediksi: {prediksi_final(s)}")

Tahap 4: Uji Coba Inference
Input: gamenya seru banget -> Prediksi: Positif
Input: banyak cit curang -> Prediksi: Negatif
Input: lumayan buat ngisi waktu -> Prediksi: Netral
